# 02 · Feature Engineering

Computes all **15 parameters** grouped into three families.

**Performance** — Alpha, Sharpe, Rolling Returns, Sortino, Treynor, AUM, Capture Ratio  
**Cost** — Turnover, TER, Management Fees, Transaction Cost, Load Fees  
**Risk** — Beta, Std Dev, VaR

Annualization uses 252 trading days, risk-free rate ≈ 6.5% (Indian 10-yr G-Sec).

In [ ]:
# Bootstrap path so the notebook finds the src/ package
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)


In [ ]:
from data_loader import load_all, LoaderConfig
from feature_engineering import compute_fund_features

data = load_all(LoaderConfig(offline_mode=True, lookback_days=1260))
features = compute_fund_features(data["universe"], data["navs"], data["benchmarks"])
features.shape

### Performance metrics

In [ ]:
cols = ["scheme_name","category","annualized_return","sharpe_ratio","sortino_ratio","treynor_ratio","alpha","capture_ratio"]
features[cols].sort_values("sharpe_ratio", ascending=False).head(10)

### Risk metrics

In [ ]:
cols = ["scheme_name","category","beta","std_dev","var_95_historical","max_drawdown"]
features[cols].sort_values("std_dev").head(10)

### Cost metrics

In [ ]:
cols = ["scheme_name","category","ter","management_fee","transaction_cost","exit_load","turnover_ratio"]
features[cols].sort_values("ter").head(10)

### Correlation across parameters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = ["sharpe_ratio","sortino_ratio","alpha","beta","std_dev","var_95_historical","max_drawdown","ter","turnover_ratio"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(features[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, cbar_kws={"shrink":0.7}, ax=ax)
ax.set_title("Parameter correlation matrix")
plt.tight_layout(); plt.show()

### Within-category z-scores

Each metric is standardized within its fund category, so a large-cap fund is compared against other large-caps — not against debt funds.

In [ ]:
zcols = [c for c in features.columns if c.endswith("_z")]
features[["scheme_name","category"] + zcols].head(8)

### Persist the feature matrix

In [ ]:
from pathlib import Path
out = Path.cwd().parent / "data" / "processed" / "features.parquet"
features.to_parquet(out, index=False)
print("saved:", out)